# Country pairwise classification delta

In [ ]:
from __future__ import annotations

import json
import math
from datetime import date
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from matplotlib.cm import ScalarMappable
from shapely.geometry import box

from liljegren_analysis.country_cells import (
    cached_natural_earth_boundaries,
    country_cells,
    load_countries,
)

In [ ]:
# Analysis parameters: edit these values before running the notebook.
COUNTRY_CODE = "JPN"  # ISO alpha-2 or alpha-3; use "JPN" for Japan.
ANALYSIS_START_DATE = date(2025, 1, 1)
ANALYSIS_END_DATE = date(2025, 12, 31)
# WBGT_THRESHOLDS_C = (31, 32, 33)  # SG MOM/NEA
WBGT_THRESHOLDS_C = (21, 25, 28, 31, 33)  # JPN MoE
DUCKDB_THREADS = 0  # Set to 0 to use DuckDB's default.

In [ ]:
working_directory = Path.cwd().resolve()
repository = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if repository is None:
    raise FileNotFoundError("Could not locate the repository root")

country_code = COUNTRY_CODE.strip().upper()
start_date = ANALYSIS_START_DATE
end_date = ANALYSIS_END_DATE
if start_date > end_date:
    raise ValueError("start_date must not be after end_date")


thresholds_c = tuple(float(value) for value in WBGT_THRESHOLDS_C)
if (
    not thresholds_c
    or any(not math.isfinite(value) for value in thresholds_c)
    or any(left >= right for left, right in zip(thresholds_c, thresholds_c[1:]))
):
    raise ValueError("thresholds_c must be finite and strictly increasing")

threads = int(DUCKDB_THREADS)
if threads < 0:
    raise ValueError("DUCKDB_THREADS must be non-negative")
{"country_code": country_code, "date_range": (start_date, end_date), "thresholds_c": thresholds_c, "threads": threads}

## Resolve country cells

In [ ]:
boundaries = cached_natural_earth_boundaries(repository / "data" / "boundaries")
country = load_countries(boundaries, [country_code])[0]
selected_cells = country_cells(country)
if not selected_cells:
    raise ValueError(f"No ERA5 cells intersect {country.name}")
output_directory = (
    repository / "data" / "country_classification" / country.iso_a3.lower()
)
country_cache = (
    repository / "data" / "country" / country.iso_a3.lower() / "observations.parquet"
)
country_cache_manifest = country_cache.with_name("manifest.json")

country_cells_frame = pd.DataFrame(
    [
        {
            "cell_id": cell.cell_id,
            "latitude": cell.latitude,
            "longitude": cell.longitude,
            "center_in_country": cell.center_in_country,
        }
        for cell in selected_cells
    ]
)
cell_ids = tuple(country_cells_frame["cell_id"])

{"country": country.name, "iso_a3": country.iso_a3, "cells": len(cell_ids)}

country_cells_frame

## Country coverage within the selected cells

In [ ]:
earth_radius_km = 6_371.0088
half_cell_degrees = 0.125


def spherical_ring_area_km2(coordinates) -> float:
    points = list(coordinates)
    signed_area = sum(
        math.radians(longitude_b - longitude_a)
        * (math.sin(math.radians(latitude_a)) + math.sin(math.radians(latitude_b)))
        for (longitude_a, latitude_a), (longitude_b, latitude_b)
        in zip(points, points[1:])
    )
    return abs(signed_area) * earth_radius_km**2 / 2


def spherical_area_km2(geometry) -> float:
    if geometry.is_empty:
        return 0.0
    if geometry.geom_type == "Polygon":
        return spherical_ring_area_km2(geometry.exterior.coords) - sum(
            spherical_ring_area_km2(interior.coords) for interior in geometry.interiors
        )
    if hasattr(geometry, "geoms"):
        return sum(spherical_area_km2(part) for part in geometry.geoms)
    return 0.0


cell_footprints = {
    cell.cell_id: box(
        cell.longitude - half_cell_degrees,
        cell.latitude - half_cell_degrees,
        cell.longitude + half_cell_degrees,
        cell.latitude + half_cell_degrees,
    )
    for cell in selected_cells
}
cell_intersections = {
    cell_id: country.geometry.intersection(footprint)
    for cell_id, footprint in cell_footprints.items()
}
coverage = pd.DataFrame(
    [
        {
            "cell_id": cell_id,
            "cell_area_km2": spherical_area_km2(footprint),
            "country_area_km2": spherical_area_km2(cell_intersections[cell_id]),
        }
        for cell_id, footprint in cell_footprints.items()
    ]
)
coverage["country_percent_of_cell"] = (
    100.0 * coverage["country_area_km2"] / coverage["cell_area_km2"]
)
country_boundary_area_km2 = spherical_area_km2(country.geometry)
coverage["percent_of_country_boundary"] = (
    100.0 * coverage["country_area_km2"] / country_boundary_area_km2
)
if not math.isclose(
    coverage["country_area_km2"].sum(),
    country_boundary_area_km2,
    rel_tol=1e-6,
):
    raise ValueError(f"Selected cells do not cover the full {country.name} boundary")

country_cells_frame = country_cells_frame.merge(
    coverage, on="cell_id", validate="one_to_one"
)

display(
    country_cells_frame[
        [
            "cell_id",
            "latitude",
            "longitude",
            "country_area_km2",
            "country_percent_of_cell",
            "percent_of_country_boundary",
        ]
    ].round(3)
)

In [ ]:
def polygon_parts(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry
    elif hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            yield from polygon_parts(part)


figure, axis = plt.subplots(figsize=(10, 8), constrained_layout=True)
color_scale = Normalize(vmin=0.0, vmax=100.0)
color_map = plt.colormaps["viridis"]
coverage_by_cell = country_cells_frame.set_index("cell_id")["country_percent_of_cell"]
for cell_id, footprint in cell_footprints.items():
    minimum_x, minimum_y, maximum_x, maximum_y = footprint.bounds
    axis.add_patch(
        Rectangle(
            (minimum_x, minimum_y),
            maximum_x - minimum_x,
            maximum_y - minimum_y,
            facecolor=color_map(color_scale(coverage_by_cell[cell_id])),
            edgecolor="white",
            linewidth=0.45,
            zorder=2,
        )
    )

for polygon in polygon_parts(country.geometry):
    x, y = polygon.exterior.xy
    axis.fill(
        x,
        y,
        facecolor="none",
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
    )

if len(country_cells_frame) <= 20:
    for row in country_cells_frame.itertuples(index=False):
        axis.text(
            row.longitude,
            row.latitude,
            f"{row.country_area_km2:.1f} km²\n{row.country_percent_of_cell:.1f}%",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if row.country_percent_of_cell < 45 else "black",
            zorder=4,
        )

minimum_longitude = min(geometry.bounds[0] for geometry in cell_footprints.values())
minimum_latitude = min(geometry.bounds[1] for geometry in cell_footprints.values())
maximum_longitude = max(geometry.bounds[2] for geometry in cell_footprints.values())
maximum_latitude = max(geometry.bounds[3] for geometry in cell_footprints.values())
longitude_padding = max(0.02, (maximum_longitude - minimum_longitude) * 0.03)
latitude_padding = max(0.02, (maximum_latitude - minimum_latitude) * 0.03)
axis.set_xlim(minimum_longitude - longitude_padding, maximum_longitude + longitude_padding)
axis.set_ylim(minimum_latitude - latitude_padding, maximum_latitude + latitude_padding)
axis.set_aspect(1 / math.cos(math.radians(country.geometry.centroid.y)))
axis.set_xlabel("Longitude (degrees east)")
axis.set_ylabel("Latitude (degrees north)")
axis.set_title(
    f"{country.name} coverage within {len(country_cells_frame):,} ERA5 0.25° cells"
)
axis.grid(alpha=0.2)
color_bar = figure.colorbar(
    ScalarMappable(norm=color_scale, cmap=color_map), ax=axis, shrink=0.8, pad=0.02
)
color_bar.set_label(f"{country.name} area (% of cell)")

plt.show()

## Load the country observation cache

In [ ]:
command = (
    f"uv run python scripts/extract_country.py {country.iso_a3} "
    f"--start-date {start_date} --end-date {end_date}"
)
if not country_cache.is_file() or not country_cache_manifest.is_file():
    raise FileNotFoundError(
        f"Country cache does not exist: {country_cache}\nGenerate it with: {command}"
    )

with country_cache_manifest.open(encoding="utf-8") as source:
    cache_manifest = json.load(source)
cache_start_date = date.fromisoformat(cache_manifest["start_date"])
cache_end_date = date.fromisoformat(cache_manifest["end_date"])
if (
    cache_manifest.get("schema_version") != 1
    or cache_manifest.get("country_iso_a3") != country.iso_a3
    or cache_start_date > start_date
    or cache_end_date < end_date
):
    raise ValueError(
        f"Country cache metadata does not cover this analysis. Regenerate with: "
        f"{command} --force"
    )

connection = duckdb.connect()
if threads > 0:
    connection.execute(f"SET threads = {threads}")
try:
    observations = (
        connection.read_parquet(str(country_cache))
        .filter(
            f"result_date >= DATE '{start_date}' "
            f"AND result_date <= DATE '{end_date}'"
        )
        .project(
            "result_date, run_id, implementation, cycle_ordinal, "
            "row_id, cell_id, row_ordinal, value_c"
        )
        .df()
    )
finally:
    connection.close()

observations.info()
display(observations.head(20))

In [ ]:
identity_columns = [
    "result_date",
    "run_id",
    "implementation",
    "cycle_ordinal",
    "row_id",
]
duplicate_count = int(observations.duplicated(identity_columns, keep=False).sum())
if duplicate_count:
    raise ValueError(f"observations contains {duplicate_count} duplicate rows")
if observations.empty:
    raise ValueError(f"No valid {country.name} observations were found")

available_cells = set(observations["cell_id"].unique())
country_cells_frame["present_in_campaign"] = country_cells_frame["cell_id"].isin(available_cells)
observation_summary = pd.DataFrame(
    {
        "valid_observations": [len(observations)],
        "implementations": [observations["implementation"].nunique()],
        "cells": [observations["cell_id"].nunique()],
        "first_date": [observations["result_date"].min()],
        "last_date": [observations["result_date"].max()],
    }
)
display(country_cells_frame)
display(observation_summary)

## Classify and pair

In [ ]:
bounds = (-np.inf, *thresholds_c, np.inf)
class_labels = tuple(
    (
        f"< {upper:g} C"
        if index == 0
        else f">= {lower:g} C"
        if index == len(thresholds_c)
        else f"[{lower:g}, {upper:g}) C"
    )
    for index, (lower, upper) in enumerate(zip(bounds, bounds[1:]))
)
class_definitions = pd.DataFrame(
    {"class_id": range(len(class_labels)), "class_label": class_labels}
)
class_label_by_id = dict(zip(class_definitions["class_id"], class_definitions["class_label"]))

classified = observations.copy()
classified["class_id"] = pd.cut(
    classified["value_c"],
    bins=bounds,
    labels=False,
    right=False,
).astype("int8")
classified["class_label"] = classified["class_id"].map(class_label_by_id)
display(class_definitions)
display(classified.head(20))

In [ ]:
pair_keys = ["result_date", "run_id", "cycle_ordinal", "row_id", "cell_id"]
pair_values = pair_keys + ["implementation", "value_c", "class_id"]
paired_classifications = classified[pair_values].merge(
    classified[pair_values],
    on=pair_keys,
    how="inner",
    suffixes=("_a", "_b"),
    validate="many_to_many",
)
paired_classifications = paired_classifications.loc[
    paired_classifications["implementation_a"]
    < paired_classifications["implementation_b"]
].copy()
paired_classifications["difference_c"] = (
    paired_classifications["value_c_a"]
    - paired_classifications["value_c_b"]
)
paired_classifications["class_difference"] = (
    paired_classifications["class_id_a"]
    - paired_classifications["class_id_b"]
)
paired_classifications["classification_changed"] = (
    paired_classifications["class_difference"] != 0
)
paired_classifications.reset_index(drop=True, inplace=True)
paired_classifications.info()
display(paired_classifications.head(20))

## Classification-change results

In [ ]:
pair_columns = ["implementation_a", "implementation_b"]
summary_input = paired_classifications.assign(
    a_higher_class=paired_classifications["class_difference"] > 0,
    b_higher_class=paired_classifications["class_difference"] < 0,
)
change_summary = (
    summary_input
    .groupby(pair_columns, as_index=False, sort=True)
    .agg(
        n_paired=("classification_changed", "size"),
        n_changed=("classification_changed", "sum"),
        a_higher_class=("a_higher_class", "sum"),
        b_higher_class=("b_higher_class", "sum"),
    )
)
change_summary["percent_changed"] = (
    100.0 * change_summary["n_changed"] / change_summary["n_paired"]
).round(4)
change_summary = change_summary[
    pair_columns
    + ["n_paired", "n_changed", "percent_changed", "a_higher_class", "b_higher_class"]
]
display(change_summary)

In [ ]:
transition_columns = pair_columns + ["class_id_a", "class_id_b"]
transition_counts = (
    paired_classifications
    .groupby(transition_columns, as_index=False, sort=True)
    .size()
    .rename(columns={"size": "n"})
)
transition_counts.insert(
    3,
    "class_a_label",
    transition_counts["class_id_a"].map(class_label_by_id),
)
transition_counts.insert(
    5,
    "class_b_label",
    transition_counts["class_id_b"].map(class_label_by_id),
)
transition_counts["classification_changed"] = (
    transition_counts["class_id_a"] != transition_counts["class_id_b"]
)
display(transition_counts)

In [ ]:
changed_observations = (
    paired_classifications.loc[paired_classifications["classification_changed"]]
    .assign(absolute_difference_c=lambda frame: frame["difference_c"].abs())
    .sort_values(
        ["absolute_difference_c", "result_date", "row_id"],
        ascending=[False, True, True],
    )
    .drop(columns="absolute_difference_c")
    .head(100)
)
display(changed_observations)

## Export compact classification results

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
country_cells_frame.to_csv(output_directory / "cells.csv", index=False)
class_definitions.to_csv(output_directory / "classes.csv", index=False)
change_summary.to_csv(output_directory / "pair_change_summary.csv", index=False)
transition_counts.to_csv(output_directory / "pair_transition_counts.csv", index=False)
print(f"Saved classification outputs to {output_directory}")